In [1]:
import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime
import schedule
import time

def connect_mt5():
    if not mt5.initialize():
        raise RuntimeError(" !!! Error >>> MT5 initialize() failed")
    print("Good! Connected to MT5")

def shutdown_mt5():
    mt5.shutdown()
    print("MT5 disconnected")

def get_daily_data(symbol="EURUSD", n=10):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_D1, 0, n)
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    return df

def detect_engulfing(df):
    if len(df) < 2:
        return "neutral"
    prev = df.iloc[-2]
    curr = df.iloc[-1]

    if prev['close'] < prev['open'] and curr['close'] > curr['open']:
        if curr['close'] > prev['open'] and curr['open'] < prev['close']:
            return "bullish"
    if prev['close'] > prev['open'] and curr['close'] < curr['open']:
        if curr['open'] > prev['close'] and curr['close'] < prev['open']:
            return "bearish"
    return "neutral"

def place_trade(symbol, signal, df, lot=0.1):
    # signal = "bullish" # !!! using this line just to test the function and MT5 connection
    if signal not in ["bullish", "bearish"]:
        print("No trade signal (neutral) — skipping trade.")
        return

    tick = mt5.symbol_info_tick(symbol)
    price = tick.ask if signal == "bullish" else tick.bid
    order_type = mt5.ORDER_TYPE_BUY if signal == "bullish" else mt5.ORDER_TYPE_SELL

    curr = df.iloc[-1]
    sl_price = curr['low'] if signal == "bullish" else curr['high']
    sl_dist = abs(price - sl_price)
    tp_dist = 2 * sl_dist
    sl = round(price - sl_dist, 5) if signal == "bullish" else round(price + sl_dist, 5)
    tp = round(price + tp_dist, 5) if signal == "bullish" else round(price - tp_dist, 5)

    if not mt5.symbol_select(symbol, True):
        print(f"!!! >>> Failed to select symbol: {symbol}")
        return

    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": lot,
        "type": order_type,
        "price": price,
        "sl": sl,
        "tp": tp,
        "deviation": 10,
        "magic": 234000,
        "comment": f"{signal} engulfing SL/TP",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }

    result = mt5.order_send(request)

    if result.retcode != mt5.TRADE_RETCODE_DONE:
        print(f"Order failed: {result.retcode}")
    else:
        print(f"{signal.capitalize()} trade placed successfully at {price}")
        print(f"SL: {sl} | TP: {tp}")


def run_strategy():
    weekday = datetime.now().weekday()
    # if weekday >= 5:
    #     print("Weekend — skipping execution.")
    #     return

    symbol = "EURUSD"
    connect_mt5()
    df = get_daily_data(symbol, n=5)
    print(df.tail(2))

    signal = detect_engulfing(df)
    print(f"\\\ Signal: {signal}")

    place_trade(symbol, signal, df)
    shutdown_mt5()

In [2]:
# test the strategy function
run_strategy()

Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-22  1.12307  1.12561  1.11884  1.12143        94470       0   
4 2026-10-23  1.12143  1.12264  1.11844  1.12009        82726       0   

   real_volume  
3            0  
4            0  
\\ Signal: neutral
No trade signal (neutral) — skipping trade.
MT5 disconnected


In [3]:
# Schedule the job daily at midnight
schedule.every().day.at("00:00").do(run_strategy)

print("Strategy scheduler is running...")

while True:
    schedule.run_pending()

    time.sleep(30)


Strategy scheduler is running...


KeyboardInterrupt: 

In [8]:
# Shutdown connection
mt5.shutdown()

True

In [ ]:
schedule.every(30).seconds.do(run_strategy)

print("------> Strategy scheduler is running every 30 seconds...")

while True:
    schedule.run_pending()
    time.sleep(1)

------> Strategy scheduler is running every 10 seconds...
Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11216        12098       0   

   real_volume  
3            0  
4            0  
\\ Signal: neutral
No trade signal (neutral) — skipping trade.
MT5 disconnected
Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11215        12101       0   

   real_volume  
3            0  
4            0  
\\ Signal: neutral
No trade signal (neutral) — skipping trade.
MT5 disconnected
Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.

Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11218        12178       0   

   real_volume  
3            0  
4            0  
\\ Signal: neutral
No trade signal (neutral) — skipping trade.
MT5 disconnected
Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11218        12182       0   

   real_volume  
3            0  
4            0  
\\ Signal: neutral
No trade signal (neutral) — skipping trade.
MT5 disconnected
Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11215        12184       0   

   real_v

Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11214        12255       0   

   real_volume  
3            0  
4            0  
\\ Signal: neutral
No trade signal (neutral) — skipping trade.
MT5 disconnected
Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11214        12259       0   

   real_volume  
3            0  
4            0  
\\ Signal: neutral
No trade signal (neutral) — skipping trade.
MT5 disconnected
Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11214        12259       0   

   real_v

Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11213        12316       0   

   real_volume  
3            0  
4            0  
\\ Signal: neutral
No trade signal (neutral) — skipping trade.
MT5 disconnected
Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11215        12323       0   

   real_volume  
3            0  
4            0  
\\ Signal: neutral
No trade signal (neutral) — skipping trade.
MT5 disconnected
Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11214        12324       0   

   real_v

Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11211        12816       0   

   real_volume  
3            0  
4            0  
\\ Signal: neutral
No trade signal (neutral) — skipping trade.
MT5 disconnected
Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11208        12831       0   

   real_volume  
3            0  
4            0  
\\ Signal: neutral
No trade signal (neutral) — skipping trade.
MT5 disconnected
Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11208        12831       0   

   real_v

Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11204        12944       0   

   real_volume  
3            0  
4            0  
\\ Signal: neutral
No trade signal (neutral) — skipping trade.
MT5 disconnected
Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11203        12949       0   

   real_volume  
3            0  
4            0  
\\ Signal: neutral
No trade signal (neutral) — skipping trade.
MT5 disconnected
Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11199  1.11200        12952       0   

   real_v

Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11178  1.11183        13059       0   

   real_volume  
3            0  
4            0  
\\ Signal: neutral
No trade signal (neutral) — skipping trade.
MT5 disconnected
Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11177  1.11178        13071       0   

   real_volume  
3            0  
4            0  
\\ Signal: neutral
No trade signal (neutral) — skipping trade.
MT5 disconnected
Good! Connected to MT5
        time     open     high      low    close  tick_volume  spread  \
3 2026-10-27  1.11733  1.11791  1.11223  1.11242        63907       0   
4 2026-10-28  1.11230  1.11320  1.11177  1.11178        13071       0   

   real_v

In [2]:
!pip install MetaTrader5


In [4]:
!pip install schedule